# Validación offline del modelo

Este notebook se ejecuta desde la carpeta del modelo y no usa Internet.

In [1]:
import os

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

In [2]:
from pathlib import Path
import json
import os
import sys
import platform
import importlib.metadata

def resolve_model_path():
    configured = os.environ.get('MODEL_PATH', '').strip()
    if configured:
        return Path(configured).expanduser().resolve()
    if os.environ.get('DATABRICKS_RUNTIME_VERSION'):
        try:
            dbutils.widgets.text('model_path', '')
            configured = dbutils.widgets.get('model_path').strip()
        except Exception:
            configured = ''
        if configured:
            return Path(configured).resolve()
        raise RuntimeError('En Databricks indique MODEL_PATH o el widget model_path con una ruta /Volumes/...')
    working_directory = Path.cwd().resolve()
    if (working_directory / 'model-metadata.json').is_file():
        return working_directory
    relative_model = Path('models') / 'embeddings' / '.distiluse-base-multilingual-cased-v2.partial-1d4e0d0fb1474ff697afb31fe418757b'
    for root in (working_directory, *working_directory.parents):
        candidate = root / relative_model
        if (candidate / 'model-metadata.json').is_file():
            return candidate.resolve()
    raise RuntimeError(
        'No se encontró la carpeta del modelo. Ejecute desde el repositorio o defina MODEL_PATH.'
    )

MODEL_PATH = resolve_model_path()
runtime = os.environ.get('DATABRICKS_RUNTIME_VERSION', 'local')
print('PREFLIGHT')
print(f'Python: {sys.version.split()[0]}')
print(f'DATABRICKS_RUNTIME_VERSION: {runtime}')
# DATABRICKS_RUNTIME_VERSION no distingue Runtime estándar de ML.
# Se valida la capacidad requerida comprobando los paquetes instalados.
for package in ('torch', 'transformers', 'sentence-transformers'):
    try:
        print(f'{package}: {importlib.metadata.version(package)}')
    except importlib.metadata.PackageNotFoundError as exc:
        raise RuntimeError(f'Falta {package}; use ML Runtime. No se puede instalar desde PyPI.') from exc
try:
    assert MODEL_PATH.is_dir(), f'No existe MODEL_PATH: {MODEL_PATH}'
    next(MODEL_PATH.iterdir(), None)
except PermissionError as exc:
    raise RuntimeError('Sin acceso al Volume. En SINGLE_USER se requieren USE CATALOG, USE SCHEMA y READ VOLUME para el principal del cluster.') from exc
LFS_PREFIX = b'version https://git-lfs.github.com/spec/v1'
for artifact in MODEL_PATH.rglob('*'):
    if artifact.is_file():
        with artifact.open('rb') as handle:
            if handle.read(len(LFS_PREFIX)).startswith(LFS_PREFIX):
                raise RuntimeError(f'Puntero Git LFS detectado: {artifact.name}. Ejecute git lfs pull.')
print('PREFLIGHT OK')
metadata_path = MODEL_PATH / 'model-metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
required_fields = {'name', 'model_type', 'source', 'revision', 'framework', 'python_target'}
missing_fields = required_fields - metadata.keys()
assert not missing_fields, f'Metadata incompleta: {sorted(missing_fields)}'
assert MODEL_PATH.name.startswith('.') or metadata['name'] == MODEL_PATH.name, 'El nombre no coincide con la carpeta'
assert metadata['model_type'] == 'embedding', 'Tipo de modelo incorrecto'
from packaging.specifiers import SpecifierSet
from packaging.version import Version
python_target = metadata['python_target']
if Version(platform.python_version()) not in SpecifierSet(python_target):
    print(f'ADVERTENCIA: Python {platform.python_version()} queda fuera de python_target {python_target}; la evidencia no prueba compatibilidad.')
print(f'Model path: {MODEL_PATH.resolve()}')
for field in ('name', 'model_type', 'source', 'revision', 'framework', 'python_target', 'export_environment'):
    print(f'{field}: {metadata[field]}')
print('Versiones que producen esta evidencia:')
for package in ('sentence-transformers', 'transformers', 'torch'):
    print(f'{package}: {importlib.metadata.version(package)}')
print(f'Python: {platform.python_version()}')

PREFLIGHT
Python: 3.12.3
DATABRICKS_RUNTIME_VERSION: local
torch: 2.7.0+cpu
transformers: 4.51.3
sentence-transformers: 4.0.1
PREFLIGHT OK
Model path: <local model directory>
name: distiluse-base-multilingual-cased-v2
model_type: embedding
source: sentence-transformers/distiluse-base-multilingual-cased-v2
revision: bfe45d0732ca50787611c0fe107ba278c7f3f889
framework: sentence-transformers
python_target: >=3.11,<3.13
export_environment: {'python': '3.12.3', 'sentence_transformers': '4.0.1', 'transformers': '4.51.3', 'torch': '2.7.0+cpu', 'numpy': '2.1.3'}
Versiones que producen esta evidencia:
sentence-transformers: 4.0.1
transformers: 4.51.3
torch: 2.7.0+cpu
Python: 3.12.3


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    str(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=False,
)

In [4]:
import numpy as np

texts = [
    'Cliente solicita financiamiento para capital de trabajo.',
    'La empresa presenta crecimiento sostenido de ventas.',
    'La compañía mantiene una posición financiera estable.',
]
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=False,
)
assert embeddings.shape[0] == len(texts)
assert embeddings.ndim == 2
assert embeddings.shape[1] > 0
assert np.isfinite(embeddings).all()
print(f'modelo: {metadata["name"]}')
print(f'cantidad de textos: {len(texts)}')
print(f'shape: {embeddings.shape}')
print(f'dimensión del embedding: {embeddings.shape[1]}')
print(f'dtype: {embeddings.dtype}')
print('resultado: OK')

modelo: distiluse-base-multilingual-cased-v2
cantidad de textos: 3
shape: (3, 512)
dimensión del embedding: 512
dtype: float32
resultado: OK


In [5]:
print('VALIDATION OK')
print('Modelo cargado y ejecutado usando únicamente archivos locales.')

VALIDATION OK
Modelo cargado y ejecutado usando únicamente archivos locales.
